# ConvNeXt-Tiny — Optuna HPO + Ordinal Focal Loss

Adapts the ConvNeXt-Small v2 pipeline for the **Tiny** backbone:
- **Smaller backbone** — 28M params vs 50M (Small), more VRAM-efficient per trial
- **AdamW** (ConvNeXt was designed for it) + `weight_decay` tuned via Optuna
- **`unfreeze_blocks`** added to Optuna search (1–6)
- **Richer augmentation**: RandomBrightnessContrast + HueSaturationValue + RandomRotate90 (H&E stain)
- **Pure OrdinalFocalLoss** — no conflicting MSE ordinal penalty
- **Longer patience** (15 epochs) — kappa curve is noisy, needs more room
- **VRAM cleanup** between Optuna trials (gc + empty_cache)
- **`cudnn.benchmark = True`** for stable input shapes

**Trial speedup:**
- Each trial trains on a **stratified 35% subset** of the training data (`TRIAL_SUBSET_FRAC`)
- Trials are ~3× faster → `N_TRIALS` = 60 (same wall-clock budget, 2× the search)
- `SuccessiveHalvingPruner` (ASHA) for better budget allocation under this regime
- Subset seed is tied to `trial.number` for reproducibility

In [ ]:
import os
import gc
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import optim
from torch.utils.data import DataLoader
from torch.utils.data.sampler import RandomSampler
from torch.amp import autocast, GradScaler
from torchvision.models import convnext_tiny, ConvNeXt_Tiny_Weights
import albumentations as Albu
from warmup_scheduler import GradualWarmupScheduler
from sklearn.metrics import (
    accuracy_score, cohen_kappa_score, f1_score,
    recall_score, precision_score, classification_report, confusion_matrix
)
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import optuna
from optuna.samplers import TPESampler
import sys
sys.path.append('../../../')
from utils.dataset import PandasDataset

## Configuration

In [ ]:
SEED           = 42
NUM_WORKERS    = 4
OUTPUT_CLASSES = 5       # ordinal thresholds for ISUP 0-5
WARMUP_EPOCHS  = 2
WARMUP_FACTOR  = 2
N_EPOCHS_FULL  = 60      # final training budget
N_EPOCHS_TRIAL = 20      # per-trial budget (ConvNeXt needs more warmup)
PATIENCE       = 15      # kappa is noisy — give more room
N_TRIALS       = 60      # more trials since each is ~3x faster with subset data
PATIENCE_TRIAL = 5       # per-trial early stopping
VAL_FOLD       = 3
USE_AMP        = True
TRIAL_SUBSET_FRAC = 0.35  # fraction of train data per trial (stratified)

ROOT_DIR   = '../../..'
DATA_DIR   = '../../../..'
IMAGES_DIR = os.path.join(DATA_DIR, 'tiles')

os.makedirs('logs', exist_ok=True)
os.makedirs('models', exist_ok=True)

MODEL_PATH = 'models/convnext-tiny-optuna.pth'
LOG_PATH   = 'logs/convnext-tiny-optuna.txt'
OPTUNA_DB  = 'logs/convnext-tiny-optuna.db'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True

print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')

## VRAM Utilities

In [13]:
def free_vram(*objs):
    for o in objs:
        try:
            del o
        except Exception:
            pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


def vram_report(tag=''):
    if not torch.cuda.is_available():
        return
    alloc    = torch.cuda.memory_allocated() / 1e9
    reserved = torch.cuda.memory_reserved() / 1e9
    peak     = torch.cuda.max_memory_allocated() / 1e9
    print(f'  [VRAM {tag}] alloc={alloc:.2f}GB reserved={reserved:.2f}GB peak={peak:.2f}GB')

## Loss Function — Pure OrdinalFocalLoss

Removed the extra MSE ordinal penalty from v1 — it introduced conflicting gradients and required tuning two extra hyperparameters (alpha, beta) simultaneously.

In [14]:
class OrdinalFocalLoss(nn.Module):

    def __init__(
        self,
        alpha: float = 0.25,
        gamma: float = 2.0,
        ordinal_weight: float = 0.2,
        reduction: str = 'mean'
    ):
        super().__init__()

        self.alpha = alpha
        self.gamma = gamma
        self.ordinal_weight = ordinal_weight
        self.reduction = reduction

    def forward(
        self,
        logits: torch.Tensor,
        targets: torch.Tensor
    ) -> torch.Tensor:

        # numerical stability under AMP
        logits = logits.float()
        targets = targets.float()

        probs = torch.sigmoid(logits)

        # BCE
        bce = F.binary_cross_entropy_with_logits(
            logits,
            targets,
            reduction='none'
        )

        # focal term
        p_t = probs * targets + (1 - probs) * (1 - targets)

        loss = self.alpha * ((1 - p_t) ** self.gamma) * bce

        # reduction
        if self.reduction == 'mean':
            focal_loss = loss.mean()
        elif self.reduction == 'sum':
            focal_loss = loss.sum()
        else:
            focal_loss = loss

        # ordinal penalty
        expected_class = probs.sum(dim=1)
        target_class = targets.sum(dim=1)

        max_class = logits.shape[1]

        ordinal_loss = (
            (expected_class - target_class) ** 2
        ).mean() / (max_class ** 2)

        # final loss
        total_loss = focal_loss + (
            self.ordinal_weight * ordinal_loss
        )

        return total_loss

def decode_ordinal_predictions(logits: torch.Tensor) -> torch.Tensor:
    return (torch.sigmoid(logits.float()) > 0.5).sum(dim=1)


## ConvNeXt-Tiny Wrapper

`unfreeze_last_blocks` is tuned by Optuna (1–6).

In [ ]:
class ConvNeXtTiny(nn.Module):
    """
    ConvNeXt-Tiny wrapper with tunable unfreezing and head.

    ConvNeXt features layout (tiny):
      features[0] — stem (downsampling)
      features[1] — stage 1 (3 blocks)
      features[2] — downsampling
      features[3] — stage 2 (3 blocks)
      features[4] — downsampling
      features[5] — stage 3 (9 blocks)  ← main capacity
      features[6] — downsampling
      features[7] — stage 4 (3 blocks)   ← highest semantic level
    """
    def __init__(
        self,
        model: nn.Module,
        output_dimensions: int,
        dropout_rate: float = 0.4,
        unfreeze_blocks: int = 2,
    ):
        super().__init__()
        self.model = model

        # Freeze all backbone params
        for param in self.model.parameters():
            param.requires_grad = False

        # Unfreeze the last N feature blocks
        if hasattr(self.model, 'features') and unfreeze_blocks > 0:
            for block in self.model.features[-unfreeze_blocks:]:
                for param in block.parameters():
                    param.requires_grad = True

        # Resolve head input features
        if isinstance(self.model.classifier, nn.Sequential):
            in_features = self.model.classifier[-1].in_features
        else:
            in_features = self.model.classifier.in_features

        self.model.classifier = nn.Identity()

        self.head = nn.Sequential(
            nn.LayerNorm(in_features),
            nn.Dropout(dropout_rate),
            nn.Linear(in_features, output_dimensions),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.model(x)
        if x.ndim == 4:
            x = x.mean(dim=[2, 3])
        return self.head(x)

## Data Loading with Entropy Filtering

In [16]:
df_all = pd.read_csv(f'{ROOT_DIR}/data/train_5fold.csv')
df_all.columns = df_all.columns.str.strip()
print(f'Total records: {len(df_all)}')

df_entropy = pd.read_csv(f'{ROOT_DIR}/data/entropy.csv')
df_entropy_sorted = df_entropy.sort_values('difficulty_score', ascending=False)
n_remove  = int(len(df_entropy_sorted) * 0.20)
noisy_ids = set(df_entropy_sorted.head(n_remove)['image_id'])
df_all = df_all[~df_all['image_id'].isin(noisy_ids)].reset_index(drop=True)
print(f'After entropy filter: {len(df_all)}')


def drop_missing(df, images_dir):
    exists = df['image_id'].apply(lambda x: os.path.isfile(os.path.join(images_dir, f'{x}.png')))
    return df[exists].reset_index(drop=True)


train_idx = df_all['fold'] != VAL_FOLD
df_train  = drop_missing(df_all[train_idx].reset_index(drop=True), IMAGES_DIR)
df_val    = drop_missing(df_all[~train_idx].reset_index(drop=True), IMAGES_DIR)
df_test   = drop_missing(pd.read_csv(f'{ROOT_DIR}/data/test.csv'), IMAGES_DIR)

print(f'Train: {len(df_train)}  Val: {len(df_val)}  Test: {len(df_test)}')
print('Train class distribution:')
print(df_train['isup_grade'].value_counts().sort_index())

Total records: 9024
After entropy filter: 8844
Train: 7073  Val: 1767  Test: 1590
Train class distribution:
isup_grade
0    1953
1    1802
2     899
3     826
4     832
5     761
Name: count, dtype: int64


## Augmentation

Added `RandomBrightnessContrast` and `HueSaturationValue` — critical for H&E stain variation.
`RandomRotate90` is important since histopathology tiles have no canonical orientation.

In [17]:
train_transforms = Albu.Compose([
    Albu.Transpose(p=0.5),
    Albu.VerticalFlip(p=0.5),
    Albu.HorizontalFlip(p=0.5),
    Albu.RandomRotate90(p=0.5),
    Albu.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.4),
    Albu.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=10, p=0.3),
])

val_transforms = None

## Training and Validation Helpers

In [ ]:
def make_stratified_subset(df: pd.DataFrame, frac: float, seed: int) -> pd.DataFrame:
    """Sample a stratified fraction of df, preserving isup_grade distribution."""
    return (
        df.groupby('isup_grade', group_keys=False)
        .apply(lambda x: x.sample(frac=frac, random_state=seed))
        .reset_index(drop=True)
    )


def make_loaders(batch_size: int):
    train_ds = PandasDataset(IMAGES_DIR, df_train, transforms=train_transforms, format='png')
    val_ds   = PandasDataset(IMAGES_DIR, df_val,   transforms=val_transforms,   format='png')
    test_ds  = PandasDataset(IMAGES_DIR, df_test,  transforms=val_transforms,   format='png')

    common = dict(
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=(NUM_WORKERS > 0),
        prefetch_factor=2 if NUM_WORKERS > 0 else None,
    )
    train_loader = DataLoader(train_ds, batch_size=batch_size,
                              sampler=RandomSampler(train_ds), **common)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size,
                              sampler=RandomSampler(val_ds),   **common)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size,
                              shuffle=False, **common)
    return train_loader, val_loader, test_loader


def make_trial_loaders(batch_size: int, trial_seed: int):
    """Like make_loaders but trains on a stratified subset to speed up each trial."""
    df_subset = make_stratified_subset(df_train, TRIAL_SUBSET_FRAC, seed=trial_seed)
    train_ds  = PandasDataset(IMAGES_DIR, df_subset, transforms=train_transforms, format='png')
    val_ds    = PandasDataset(IMAGES_DIR, df_val,    transforms=val_transforms,   format='png')

    common = dict(
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=(NUM_WORKERS > 0),
        prefetch_factor=2 if NUM_WORKERS > 0 else None,
    )
    train_loader = DataLoader(train_ds, batch_size=batch_size,
                              sampler=RandomSampler(train_ds), **common)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, **common)
    return train_loader, val_loader


def run_epoch_train(model, loader, optimizer, loss_fn, device, scaler, accum_steps=4):
    model.train()
    losses = []
    optimizer.zero_grad(set_to_none=True)

    for step, (imgs, targets, _) in enumerate(tqdm(loader, desc='Train', leave=False)):
        imgs    = imgs.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        with autocast(device_type='cuda', enabled=USE_AMP, dtype=torch.float16):
            logits = model(imgs)
            loss   = loss_fn(logits, targets) / accum_steps

        scaler.scale(loss).backward()

        if (step + 1) % accum_steps == 0 or (step + 1) == len(loader):
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        losses.append(loss.item() * accum_steps)

    return float(np.mean(losses))


def run_epoch_val(model, loader, loss_fn, device):
    model.eval()
    losses, preds, gts = [], [], []
    with torch.no_grad():
        for imgs, targets, _ in tqdm(loader, desc='Val', leave=False):
            imgs    = imgs.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)
            with autocast(device_type='cuda', enabled=USE_AMP, dtype=torch.float16):
                logits = model(imgs)
                loss   = loss_fn(logits, targets)
            losses.append(loss.item())
            preds.append(decode_ordinal_predictions(logits).cpu())
            gts.append(targets.sum(1).long().cpu())

    preds = torch.cat(preds).numpy()
    gts   = torch.cat(gts).numpy()
    return dict(
        val_loss=float(np.mean(losses)),
        val_kappa=cohen_kappa_score(gts, preds, weights='quadratic'),
        val_acc=accuracy_score(gts, preds),
        val_f1=f1_score(gts, preds, average='macro', zero_division=0),
    )

## Optuna Objective

Search space:
- `lr` (1e-5 – 5e-4, log scale)
- `dropout_rate` (0.2–0.6)
- `focal_gamma` (0.5–4.0) and `focal_alpha` (0.1–0.5)
- `unfreeze_blocks` (1–6)
- `batch_size` (4 or 8)
- `weight_decay` (1e-5–0.1, log scale)

In [ ]:
def build_model(dropout_rate: float, unfreeze_blocks: int) -> nn.Module:
    backbone = convnext_tiny(weights=ConvNeXt_Tiny_Weights.DEFAULT)
    return ConvNeXtTiny(
        model=backbone,
        output_dimensions=OUTPUT_CLASSES,
        dropout_rate=dropout_rate,
        unfreeze_blocks=unfreeze_blocks,
    ).to(device)


def objective(trial: optuna.Trial) -> float:
    model = optimizer = scheduler = scheduler_cos = scaler = None
    train_loader = val_loader = None

    try:
        lr              = trial.suggest_float('lr',              1e-5, 5e-4, log=True)
        dropout_rate    = trial.suggest_float('dropout_rate',    0.2,  0.6)
        focal_gamma     = trial.suggest_float('focal_gamma',     0.5,  4.0)
        focal_alpha     = trial.suggest_float('focal_alpha',     0.1,  0.5)
        unfreeze_blocks = trial.suggest_int  ('unfreeze_blocks', 1,    6)
        batch_size      = trial.suggest_categorical('batch_size', [4, 8])
        weight_decay    = trial.suggest_float('weight_decay',    1e-5, 0.1, log=True)

        model   = build_model(dropout_rate, unfreeze_blocks)
        loss_fn = OrdinalFocalLoss(alpha=focal_alpha, gamma=focal_gamma)
        scaler  = GradScaler(enabled=USE_AMP)

        # Use a stratified subset of training data; seed per trial for reproducibility
        train_loader, val_loader = make_trial_loaders(batch_size, trial_seed=trial.number)

        optimizer = optim.AdamW(
            [p for p in model.parameters() if p.requires_grad],
            lr=lr / WARMUP_FACTOR,
            weight_decay=weight_decay,
        )
        scheduler_cos = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=max(1, N_EPOCHS_TRIAL - WARMUP_EPOCHS)
        )
        scheduler = GradualWarmupScheduler(
            optimizer, multiplier=WARMUP_FACTOR,
            total_epoch=WARMUP_EPOCHS, after_scheduler=scheduler_cos
        )

        best_kappa = -1.0
        no_improve = 0

        for epoch in range(1, N_EPOCHS_TRIAL + 1):
            run_epoch_train(model, train_loader, optimizer, loss_fn, device, scaler)
            metrics = run_epoch_val(model, val_loader, loss_fn, device)
            scheduler.step()

            kappa = metrics['val_kappa']
            trial.report(kappa, epoch)

            if trial.should_prune():
                raise optuna.exceptions.TrialPruned()

            if kappa > best_kappa:
                best_kappa = kappa
                no_improve = 0
            else:
                no_improve += 1

            if no_improve >= PATIENCE_TRIAL:
                break

        return best_kappa

    finally:
        free_vram(model, optimizer, scheduler, scheduler_cos, scaler,
                  train_loader, val_loader)

## Run Optuna Study

In [ ]:
sampler = TPESampler(seed=SEED)

# SuccessiveHalvingPruner (ASHA): allocates more budget to promising trials.
# Works well with the subset regime since early epochs on small data are cheap signal.
pruner = optuna.pruners.SuccessiveHalvingPruner(
    min_resource=3,        # start pruning from epoch 3
    reduction_factor=3,    # keep top 1/3 at each rung
    min_early_stopping_rate=1,
)

study = optuna.create_study(
    direction='maximize',
    sampler=sampler,
    pruner=pruner,
    storage=f'sqlite:///{OPTUNA_DB}',
    study_name='convnext-tiny',
    load_if_exists=True,
)

study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True, gc_after_trial=True)

print('\nBest trial:')
best = study.best_trial
print(f'  Value (QWK): {best.value:.4f}')
print('  Params:')
for k, v in best.params.items():
    print(f'    {k}: {v}')

## Optuna Visualizations

In [ ]:
from optuna.visualization.matplotlib import plot_optimization_history, plot_param_importances

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
plt.sca(axes[0]); plot_optimization_history(study); axes[0].set_title('Optimization History')
plt.sca(axes[1]); plot_param_importances(study);    axes[1].set_title('Hyperparameter Importance')
plt.tight_layout()
plt.savefig('logs/convnext-tiny-optuna-study.png', dpi=150, bbox_inches='tight')
plt.show()

## Full Training with Best Hyperparameters

In [ ]:
free_vram()

best_params = study.best_params
print('Best hyperparameters:', best_params)

model   = build_model(best_params['dropout_rate'], best_params['unfreeze_blocks'])
loss_fn = OrdinalFocalLoss(alpha=best_params['focal_alpha'], gamma=best_params['focal_gamma'])
scaler  = GradScaler(enabled=USE_AMP)

train_loader, val_loader, test_loader = make_loaders(best_params['batch_size'])

optimizer = optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=best_params['lr'] / WARMUP_FACTOR,
    weight_decay=best_params['weight_decay'],
)
scheduler_cos = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=N_EPOCHS_FULL - WARMUP_EPOCHS
)
scheduler = GradualWarmupScheduler(
    optimizer, multiplier=WARMUP_FACTOR,
    total_epoch=WARMUP_EPOCHS, after_scheduler=scheduler_cos
)

print(f'\nFull training with ConvNeXt-Tiny: up to {N_EPOCHS_FULL} epochs, patience={PATIENCE}\n')

In [ ]:
history = dict(train_loss=[], val_loss=[], val_kappa=[], val_acc=[], val_f1=[])
best_kappa = 0.0
best_epoch = 0
no_improve = 0

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

for epoch in range(1, N_EPOCHS_FULL + 1):
    print(f'Epoch {epoch}/{N_EPOCHS_FULL}')

    train_loss = run_epoch_train(model, train_loader, optimizer, loss_fn, device, scaler)
    metrics    = run_epoch_val(model, val_loader, loss_fn, device)
    scheduler.step()
    lr_now = optimizer.param_groups[0]['lr']

    history['train_loss'].append(train_loss)
    for k in ('val_loss', 'val_kappa', 'val_acc', 'val_f1'):
        history[k].append(metrics[k])

    print(f'  train_loss={train_loss:.5f}  val_loss={metrics["val_loss"]:.5f}  '
          f'kappa={metrics["val_kappa"]:.4f}  acc={metrics["val_acc"]*100:.2f}%  lr={lr_now:.7f}')
    vram_report(f'epoch {epoch}')

    with open(LOG_PATH, 'a') as f:
        f.write(
            f'epoch: {epoch} | lr: {lr_now:.7f} | '
            f'train_loss: {train_loss:.5f} | val_loss: {metrics["val_loss"]:.5f} | '
            f'val_kappa: {metrics["val_kappa"]:.4f} | val_acc: {metrics["val_acc"]:.4f}\n'
        )

    if metrics['val_kappa'] > best_kappa:
        best_kappa = metrics['val_kappa']
        best_epoch = epoch
        no_improve = 0
        torch.save(model.state_dict(), MODEL_PATH)
        print(f'  ** Best model saved (QWK={best_kappa:.4f})')
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f'\nEarly stopping at epoch {epoch}. Best: epoch {best_epoch} QWK={best_kappa:.4f}')
            break

print(f'\nTraining done. Best QWK: {best_kappa:.4f} at epoch {best_epoch}')

## Training Curves

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

axes[0, 0].plot(history['train_loss'], label='Train')
axes[0, 0].plot(history['val_loss'],   label='Val')
axes[0, 0].set_title('Loss'); axes[0, 0].legend(); axes[0, 0].grid(True)

axes[0, 1].plot(history['val_kappa'], color='orange', label='Val QWK')
axes[0, 1].axvline(best_epoch - 1, color='red', linestyle='--', label=f'Best epoch {best_epoch}')
axes[0, 1].set_title('Quadratic Weighted Kappa'); axes[0, 1].legend(); axes[0, 1].grid(True)

axes[1, 0].plot(history['val_acc'], color='green', label='Val Accuracy')
axes[1, 0].set_title('Accuracy'); axes[1, 0].legend(); axes[1, 0].grid(True)

axes[1, 1].plot(history['val_f1'], color='red', label='Val Macro F1')
axes[1, 1].set_title('Macro F1'); axes[1, 1].legend(); axes[1, 1].grid(True)

plt.suptitle('ConvNeXt-Tiny — Optuna + AdamW + OrdinalFocalLoss', y=1.01)
plt.tight_layout()
plt.savefig('logs/convnext-tiny-optuna-training.png', dpi=300, bbox_inches='tight')
plt.show()

## Test-Set Evaluation with Bootstrap CI

In [ ]:
model.load_state_dict(torch.load(MODEL_PATH, weights_only=True))
model.eval()

test_preds, test_gts = [], []
with torch.no_grad():
    for imgs, targets, _ in tqdm(test_loader, desc='Test'):
        imgs = imgs.to(device, non_blocking=True)
        with autocast(enabled=USE_AMP, dtype=torch.float16):
            logits = model(imgs)
        test_preds.append(decode_ordinal_predictions(logits).cpu())
        test_gts.append(targets.sum(1).long())

test_preds = torch.cat(test_preds).numpy()
test_gts   = torch.cat(test_gts).numpy()

pt_acc   = accuracy_score(test_gts, test_preds)
pt_kappa = cohen_kappa_score(test_gts, test_preds, weights='quadratic')
pt_f1    = f1_score(test_gts, test_preds, average='macro', zero_division=0)

N_BOOT = 1000
rng    = np.random.default_rng(SEED)
n      = len(test_gts)
boot   = dict(acc=np.empty(N_BOOT), kappa=np.empty(N_BOOT), f1=np.empty(N_BOOT))

for i in tqdm(range(N_BOOT), desc='Bootstrap'):
    idx              = rng.integers(0, n, size=n)
    boot['acc'][i]   = accuracy_score(test_gts[idx], test_preds[idx])
    boot['kappa'][i] = cohen_kappa_score(test_gts[idx], test_preds[idx], weights='quadratic')
    boot['f1'][i]    = f1_score(test_gts[idx], test_preds[idx], average='macro', zero_division=0)

def ci(arr):
    return arr.std(ddof=1), np.percentile(arr, 2.5), np.percentile(arr, 97.5)

acc_s, acc_lo, acc_hi   = ci(boot['acc'])
kap_s, kap_lo, kap_hi   = ci(boot['kappa'])
f1_s,  f1_lo,  f1_hi    = ci(boot['f1'])

print('\n' + '='*70)
print('TEST SET RESULTS — ConvNeXt-Tiny + Ordinal Focal (Optuna)')
print('='*70)
print(f'Accuracy : {pt_acc*100:.2f}% ± {acc_s*100:.2f}%  [95% CI: {acc_lo*100:.2f}%–{acc_hi*100:.2f}%]')
print(f'QW Kappa : {pt_kappa:.4f} ± {kap_s:.4f}  [95% CI: {kap_lo:.4f}–{kap_hi:.4f}]')
print(f'Macro F1 : {pt_f1:.4f} ± {f1_s:.4f}  [95% CI: {f1_lo:.4f}–{f1_hi:.4f}]')
print('='*70)
print()
print(classification_report(test_gts, test_preds,
      target_names=[f'ISUP {i}' for i in range(6)], digits=4, zero_division=0))

## Confusion Matrices

In [ ]:
cm      = confusion_matrix(test_gts, test_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
labels  = [f'ISUP {i}' for i in range(6)]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels, yticklabels=labels, ax=axes[0])
axes[0].set_title('Confusion Matrix (counts)')
axes[0].set_ylabel('True'); axes[0].set_xlabel('Predicted')

sns.heatmap(cm_norm, annot=True, fmt='.3f', cmap='Blues',
            xticklabels=labels, yticklabels=labels, ax=axes[1])
axes[1].set_title('Confusion Matrix (normalized)')
axes[1].set_ylabel('True'); axes[1].set_xlabel('Predicted')

plt.suptitle('ConvNeXt-Tiny — OrdinalFocalLoss', y=1.01)
plt.tight_layout()
plt.savefig('logs/convnext-tiny-optuna-confusion-matrix.png', dpi=300, bbox_inches='tight')
plt.show()

## Save Results

In [ ]:
results_path = 'logs/convnext-tiny-optuna-results.txt'
with open(results_path, 'w') as f:
    f.write('ConvNeXt-Tiny + OrdinalFocalLoss (Optuna HPO)\n')
    f.write('='*70 + '\n\n')
    f.write('Best Optuna hyperparameters:\n')
    for k, v in best_params.items():
        f.write(f'  {k}: {v}\n')
    f.write(f'\nOptuna best val QWK: {study.best_value:.4f}\n\n')
    f.write(f'Bootstrap resamples: {N_BOOT}\n\n')
    f.write(f'Accuracy : {pt_acc*100:.2f}% ± {acc_s*100:.2f}%  '
            f'[95% CI: {acc_lo*100:.2f}%–{acc_hi*100:.2f}%]\n')
    f.write(f'QW Kappa : {pt_kappa:.4f} ± {kap_s:.4f}  '
            f'[95% CI: {kap_lo:.4f}–{kap_hi:.4f}]\n')
    f.write(f'Macro F1 : {pt_f1:.4f} ± {f1_s:.4f}  '
            f'[95% CI: {f1_lo:.4f}–{f1_hi:.4f}]\n\n')
    f.write('Classification Report:\n')
    f.write(classification_report(test_gts, test_preds, target_names=labels, digits=4, zero_division=0))
    f.write('\nConfusion Matrix:\n')
    f.write(str(cm) + '\n')
print(f'Results saved to {results_path}')